Now that we have the cleaned data we proceed with data analysis

In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

# Load clean data
df = pd.read_csv('../data/telco_churn_clean.csv')
conn = sqlite3.connect('../telco_churn.db')

# Convert churn to numeric for calculations
df['churn_numeric'] = df['churn'].apply(lambda x: 1 if x == 'Yes' else 0)

print(f"Loaded {df.shape[0]} rows")
print(f"Churn rate: {df['churn_numeric'].mean()*100:.1f}%")

Loaded 7032 rows
Churn rate: 26.6%


In [2]:
# Finding 1: Churn rate by tenure cohort
bins = [0, 3, 12, 24, 48, 75]
labels = ['1-3 Months', '4-12 Months', '1-2 Years', '2-4 Years', '4-6 Years']
df['tenure_cohort'] = pd.cut(df['tenure'], bins=bins, labels=labels)

cohort_churn = df.groupby('tenure_cohort', observed=False)['churn_numeric'].mean() * 100
print("=== CHURN RATE BY TENURE COHORT ===")
print(cohort_churn.round(2))

=== CHURN RATE BY TENURE COHORT ===
tenure_cohort
1-3 Months     56.80
4-12 Months    39.15
1-2 Years      28.71
2-4 Years      20.39
4-6 Years       9.51
Name: churn_numeric, dtype: float64


In [3]:
# Finding 2: Churn rate by contract type
query1 = pd.read_sql("""
    SELECT 
        contract,
        COUNT(*) as total_customers,
        SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) as churned,
        ROUND(SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as churn_rate
    FROM customers
    GROUP BY contract
    ORDER BY churn_rate DESC
""", conn)
print("=== CHURN RATE BY CONTRACT TYPE ===")
print(query1)

=== CHURN RATE BY CONTRACT TYPE ===
         contract  total_customers  churned  churn_rate
0  Month-to-month             3875     1655       42.71
1        One year             1472      166       11.28
2        Two year             1685       48        2.85
